# 4. Multiple modes and partial coherence

Run the cells in order. This demonstration uses synthetic intensities; iteration counts are illustrative, not convergence guarantees. Replace the data-generation cell with your normalized experimental intensities, support and detector mask. See [the examples guide](../docs/universal_reconstruction_examples.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "library" / "phase_retrieval_universal.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from library import phase_retrieval_universal as universal

# Small synthetic data make this notebook runnable without experimental files.
shape = (32, 32)
y, x = np.indices(shape)
supportmask = ((x - 16)**2 + (y - 16)**2 < 7**2).astype(float)
mask_pixel = np.zeros(shape, dtype=int)  # 0 measured; 1 invalid/unmeasured.
magnetic_pattern = np.tanh((x - 16) / 2) * supportmask

def to_field(obj):
    return np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(obj)))

def to_object(field):
    return np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(field)))


In [ ]:
from scipy.signal import fftconvolve
primary = supportmask * np.exp(-0.2 + 0.2j)
secondary = 0.15 * supportmask * np.exp(0.3j * x)
coherent_intensity = abs(to_field(primary))**2 + abs(to_field(secondary))**2
kernel = np.exp(-((x - 16)**2 + (y - 16)**2) / 2)
kernel /= kernel.sum()
blurred = fftconvolve(coherent_intensity, kernel, mode="same")
holograms = np.stack([blurred, blurred * 1.05])
holograms = np.clip(holograms, 0, None)
mask_pixel[15:18, 15:18] = 1


Mode factors set relative support sizes, not harmonic numbers. Detector intensity is a sum of modal intensities. Here `[1, 1]` uses equal supports with distinct random starts. Partial coherence models an additional blur kernel. Modal separation and blur can be ambiguous; use additional experimental constraints where available.

In [ ]:
recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    modes=[1], mode_initialization="support_fft",
    warmup_mode=["HAPRE", "ER"], warmup_Nit=[40, 10],
    inner_mode=["ER"], inner_Nit=[10], outer_iterations=3,
    shuffle_observations=False,
    projection_relaxation=0.3,
    final_projection_relaxation=0,
    random_seed=0,
)
recipe.update(
    projection_model="none", modes=[1, 1], mode_initialization="random_phase",
    partial_coherence=True, preserve_warmup_masked_intensity=True,
    warmup_RL_it=0, RL_it=3, RL_freq=5,
    coherence_kernel_scope="per_observation",
    final_fourier_constraint=False,
)
fields, warmup, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, supportmask,
    state_labels=["sample", "sample"], energy_labels=[780., 780.],
    polarization_coefficients=[+1, -1], illumination_labels=["beam", "beam"],
    universal_recipe=recipe,
)
assert fields.shape == (2, 2, *shape)
retrieved_total_intensity = np.sum(abs(fields[0])**2, axis=0)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for mode, ax in enumerate(axes):
    ax.imshow(abs(to_object(fields[0, mode])), cmap="magma")
    ax.set_title(f"Mode {mode + 1}")
plt.tight_layout(); plt.show()


For a mixed physical scan, choose `physical_factorized` and supply its metadata. Only the first mode enters the physical model. Set `constrain_nonphysical_modes_common=True` to share secondary modes within each energy/illumination group. Spectral SVD/rank-one retrieval does not support active RL schedules; use the general driver for partial coherence.